# C4: Non-negative Matrix Factorization (NMF)

## 1. Introduction

**Non-negative Matrix Factorization (NMF)** is a dimension reduction technique similar to PCA, but with a strict constraint: **all values must be non-negative** (>= 0).

Because of this, NMF learns **parts-based representations**. Unlike PCA, which can have negative values (canceling things out), NMF is purely additive. This makes it incredibly useful for:
1.  **Recommender Systems**: Users' tastes are additive combinations of genres.
2.  **Topic Modeling**: Articles are additive combinations of topics.
3.  **Image Processing**: Faces/Digits are built from parts (eyes, noses / strokes).

### Concept
We decompose our data matrix **V** into two smaller matrices **W** (Features) and **H** (Weights/Components) such that $V \approx W \times H$.

![NMF Matrix Factorization](https://upload.wikimedia.org/wikipedia/commons/f/f9/NMF.png)
*(Source: Wikimedia Commons - NMF decomposing a matrix into two lower-rank non-negative matrices)*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import NMF
from sklearn.preprocessing import normalize, MinMaxScaler

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

## 2. NMF for Recommender Systems (Music)

We have a dataset of users and the artists they listen to. We want to recommend new artists.
- `scrobbler-small-sample.csv`: Contains `user_offset`, `artist_offset`, `playcount`.
- `artists.csv`: Contains artist names.

In [ ]:
# Load data
df_scrobbler = pd.read_csv('../../00-data/scrobbler-small-sample.csv')
df_artists = pd.read_csv('../../00-data/artists.csv', header=None, names=['Artist'])
artist_names = df_artists['Artist'].values

# Create a Matrix: Rows = Artists, Cols = Users (or vice versa). 
# Let's do Rows = Artists, Cols = Users. NMF will learn "Artist Features" and "User Preferences".
# Wait, pivot format: index='artist_offset', columns='user_offset', values='playcount'
df_pivot = df_scrobbler.pivot(index='artist_offset', columns='user_offset', values='playcount')

# Fill NaNs with 0 (no plays)
df_pivot = df_pivot.fillna(0)
X_music = df_pivot.values

print(f"Music Matrix Shape: {X_music.shape} (Artists x Users)")

In [ ]:
# Apply NMF
# Let's look for, say, 10 latent features (Genres/Styles)
nmf_music = NMF(n_components=10, random_state=42)

# Fit to the matrix
W_music = nmf_music.fit_transform(X_music) # Artist Features
H_music = nmf_music.components_         # User Preferences

print("NMF Fitted!")

### Recommending
Now, if we pick an artist, we can find similar artists by looking at the `W` matrix (Artist Features). Artists with similar feature vectors are "musically similar".

In [ ]:
# Let's pick 'Massive Attack' (Offset 0 if in our list? Let's check)
# Note: df_pivot index is 'artist_offset'. We should verify if offsets are contiguous 0..N or gaps exist.
# For simplicity in this teaching notebook, we'll assume the rows of X_music correspond to df_artists order if offsets match.
# Actually, best to map names to row indices.

artist_index_map = {name: i for i, name in enumerate(artist_names)}

# Try to find 'Massive Attack' (Row 0 in artists.csv)
target_artist = "Massive Attack"

if target_artist in artist_index_map:
    target_idx = artist_index_map[target_artist]
    # However, our X_music rows correspond to df_pivot.index (artist_offset).
    # We need to ensure artist_offset matches the row number in df_artists.
    # Let's assume artist_offset matches the csv line number for now.
    
    # Get features for Massive Attack
    target_features = W_music[target_idx]
    
    # Calculate similarity (Dot product) with all other artists
    similarities = W_music.dot(target_features)
    
    # Get top 5
    top_indices = similarities.argsort()[::-1][:6]
    
    print(f"Artists similar to {target_artist}:")
    for idx in top_indices:
        if idx != target_idx:
            # Map back to Name. df_pivot.index[idx] gives artist_offset.
            # df_artists has simple index. Assuming artist_offset is index.
            print(f"- {artist_names[idx]}")

## 3. Topic Modeling (Wikipedia)

NMF is fantastic for finding topics in text. We use `wikipedia-vectors.csv`, which is a Term-Document matrix (or Doc-Term, we'll check).

In [ ]:
# Load Vectors (Counts)
# First column is likely index. Let's load index_col=0
df_wiki = pd.read_csv('../../00-data/wikipedia-vectors.csv', index_col=0)

# Rows = Words, Cols = Articles. 
# NMF (sklearn) usually likes Samples (Articles) x Features (Words). So let's TRANSPOSE.
articles = df_wiki.columns.values
X_wiki = df_wiki.T.values # Shape: (Articles x Words)

# Load Vocabulary (Words)
with open('../../00-data/wikipedia-vocabulary-utf8.txt', 'r') as f:
    words = f.read().splitlines()

print(f"Wiki Matrix: {X_wiki.shape} (Articles x Words)")
print(f"Vocabulary Size: {len(words)}")

In [ ]:
# Fit NMF to find, say, 6 topics
nmf_wiki = NMF(n_components=6, random_state=42)
nmf_wiki.fit(X_wiki)

# W: Article-Topic Matrix (How much does each Article belong to a Topic?)
# H: Topic-Word Matrix (Which words define a Topic?)
H_wiki = nmf_wiki.components_

# Display Topics
for i, component in enumerate(H_wiki):
    top_indices = component.argsort()[::-1][:10]
    top_words = [words[k] for k in top_indices]
    print(f"Topic {i}: {top_words}")

**Observation**: You should see clear themes! E.g., one topic might be ['film', 'movie', 'actor'], another ['climate', 'warming', 'gas'].

## 4. NMF for Images (Parts-Based Learning)

Finally, let's look at `lcd_digits.csv` again. NMF learns the "strokes" that make up the digits.

In [ ]:
# Load Digits (Need to skip bad header row if consistent with C3 fix)
df_digits = pd.read_csv('../../00-data/lcd_digits.csv', header=None, skiprows=[0])
X_digits = df_digits.values

# NMF with, say, 7 components (7-segment display logic?)
nmf_digits = NMF(n_components=7, random_state=42)
nmf_digits.fit(X_digits)

# Components (The "Strokes")
components = nmf_digits.components_

# Plotting
plt.figure(figsize=(15, 3))
for i in range(7):
    plt.subplot(1, 7, i+1)
    # Reshape 104 features -> 13x8 image
    plt.imshow(components[i].reshape(13, 8), cmap='gray_r')
    plt.title(f'Component {i}')
    plt.axis('off')
plt.show()

**Interpretation**: Unlike PCA eigen-digits (which look like ghostly complete digits), NMF components look like pieces: a vertical line, a loop, a horizontal bar.

## 5. Exercises

**Task**: Inspect the Wikipedia Topic Model results for a specific article.
1.  Transform the `X_wiki` using the fitted `nmf_wiki` to get the `W_wiki` (Article-Topic) matrix.
2.  Find the index for the article "Cristiano Ronaldo".
3.  Print his Topic weights. Which topic is he most associated with?

In [ ]:
# Solution Space
# Your code here
